In [ ]:
import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch, butter, filtfilt, iirnotch

# ==========================================
# 1. SETUP PARAMETERS & LOAD DATA
# ==========================================
# Filepath to your local SEED dataset file (modify paths as needed)
file_path = r"C:\users\krish\Downloads\SEED_Dataset\Preprocessed_EEG\1_20131027.mat"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Please confirm your SEED dataset file exists at: {file_path}")

# Load raw .mat data
data = scipy.io.loadmat(file_path)
# Grab Trial 1 raw data (62 channels x total_samples)
eeg_raw = data['djc_eeg1']
fs = 250.0  # SEED sampling rate is 250 Hz

print("Successfully loaded EEG data.")
print(f"EEG raw shape: {eeg_raw.shape} (62 channels, {eeg_raw.shape[1]} samples)")

# Choose an active channel to analyze (e.g., frontal electrode where attention is prominent)
channel_idx = 0  # Channel 1
signal = eeg_raw[channel_idx]

# ==========================================
# 2. DIGITAL FILTER DESIGN (Slide 4 Preprocessing)
# ==========================================
def butter_highpass_filter(data, cutoff, fs, order=4):
    """Applies a highpass filter to eliminate DC drift."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

def notch_filter(data, notch_freq, Q, fs):
    """Applies a Notch filter to strip away Power Line Interference (50Hz or 60Hz)."""
    nyq = 0.5 * fs
    w0 = notch_freq / nyq
    b, a = iirnotch(w0, Q)
    y = filtfilt(b, a, data)
    return y

# A. Apply High Pass filter only (cutoff at 0.3 Hz as per literature)
highpass_only = butter_highpass_filter(signal, cutoff=0.3, fs=fs)

# B. Apply Line Noise Notch Filter (SEED dataset contains noise spikes around 50Hz and its harmonics)
# We will target 50 Hz power line noise
line_noise_removed = notch_filter(highpass_only, notch_freq=50.0, Q=30.0, fs=fs)
# Also apply bandpass filtering between 0.3Hz and 70Hz as specified in dataset paper
b, a = butter(4, [0.3 / (0.5*fs), 70.0 / (0.5*fs)], btype='bandpass')
fully_filtered = filtfilt(b, a, line_noise_removed)


# ==========================================
# 3. COMPUTE POWER SPECTRAL DENSITY (PSD)
# ==========================================
# Calculate Welch's periodogram to analyze signals in the Frequency Domain
f_hp, psd_hp = welch(highpass_only, fs=fs, nperseg=1024)
f_clean, psd_clean = welch(fully_filtered, fs=fs, nperseg=1024)

# Convert PSD to decibels (dB) for standard log scale plotting
psd_hp_db = 10 * np.log10(psd_hp + 1e-12)
psd_clean_db = 10 * np.log10(psd_clean + 1e-12)


# ==========================================
# 4. PLOT 1: PREPROCESSING NOISE ANALYSIS (Page 4 Graph)
# ==========================================
plt.figure(figsize=(12, 6))
plt.plot(f_hp, psd_hp_db, label='High Pass Only (0.3 Hz)', color='black', alpha=0.8, linewidth=1.5)
plt.plot(f_clean, psd_clean_db, label='Line Noise Removed (Notched 50Hz + Bandpass)', color='red', alpha=0.9, linewidth=1.2)

# Point out where the noise spike was neutralized
plt.axvline(x=50, color='blue', linestyle='--', alpha=0.5, label='Line Frequency Interference (50 Hz)')
plt.text(52, max(psd_hp_db) - 10, 'Notch Target (50Hz Noise)', color='blue', fontsize=10, fontweight='bold')

plt.title('Data Preprocessing: Power Line Interference Elimination', fontsize=14, fontweight='bold')
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Power Spectral Density (dB/Hz)', fontsize=12)
plt.xlim(0, 125)  # Nyquist frequency is 125 Hz (fs/2)
plt.ylim(min(psd_clean_db) - 5, max(psd_hp_db) + 5)
plt.legend(loc='upper right', frameon=True)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


# ==========================================
# 5. PLOT 2: SPECIFY FREQUENCY BANDS (Slide 5 Feature Engineering)
# ==========================================
# Define standard EEG frequency bands
bands = {
    'Delta (1-4 Hz)': (1, 4, '#1f77b4'),
    'Theta (4-8 Hz)': (4, 8, '#ff7f0e'),
    'Alpha (8-12 Hz)': (8, 12, '#2ca02c'),
    'Beta (12-30 Hz)': (12, 30, '#d62728'),
    'Gamma (30-45 Hz)': (30, 45, '#9467bd')
}

plt.figure(figsize=(12, 6))
plt.plot(f_clean, psd_clean, color='dimgray', linewidth=2, label='Filtered EEG Signal')

# Fill color zones under the PSD graph to visualize each neural band
for band_name, (low, high, color) in bands.items():
    idx_band = np.logical_and(f_clean >= low, f_clean <= high)
    plt.fill_between(f_clean[idx_band], psd_clean[idx_band], color=color, alpha=0.4, label=band_name)

plt.title('EEG Spectral Bands Decomposition', fontsize=14, fontweight='bold')
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Power Spectral Density ($V^2$/Hz)', fontsize=12)
plt.xlim(0.5, 50)  # Focus on the cognitive band ranges
plt.ylim(0, max(psd_clean[(f_clean >= 0.5) & (f_clean <= 50)]) * 1.2)
plt.legend(title="Spectral Ranges", loc='upper right')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()


# ==========================================
# 6. ESTIMATE ATTENTION SCORE OVER TIME (Page 5 Formula)
# ==========================================
def compute_attention_score_over_time(signal, fs, window_sec=4, step_sec=1):
    """
    Computes Beta/Alpha attention ratios dynamically using sliding windows.
    Matches: Attention Score = Beta / Alpha
    """
    # Safe fallback for np.trapz removal in NumPy 2.x
    if hasattr(np, 'trapezoid'):
        integrate_fn = np.trapezoid
    else:
        integrate_fn = np.trapz

    window_samples = int(window_sec * fs)
    step_samples = int(step_sec * fs)
    total_samples = len(signal)

    time_stamps = []
    attention_scores = []

    # Slide across the time series
    for start in range(0, total_samples - window_samples, step_samples):
        end = start + window_samples
        chunk = signal[start:end]

        # Calculate PSD for this window
        f_win, psd_win = welch(chunk, fs=fs, nperseg=window_samples)

        # Extract power in bands via simple integration
        alpha_idx = np.logical_and(f_win >= 8.0, f_win <= 12.0)
        beta_idx = np.logical_and(f_win >= 12.0, f_win <= 30.0)

        alpha_power = integrate_fn(psd_win[alpha_idx], f_win[alpha_idx])
        beta_power = integrate_fn(psd_win[beta_idx], f_win[beta_idx])

        # Avoid DivisionByZero errors
        attention_ratio = beta_power / (alpha_power + 1e-12)

        time_stamps.append((start + window_samples / 2) / fs)
        attention_scores.append(attention_ratio)

    return np.array(time_stamps), np.array(attention_scores)

# Compute the time-varying attention trajectory for the learner
times, attention_trajectory = compute_attention_score_over_time(fully_filtered, fs)

# PLOT 3: ATTENTION TRAJECTORY OVER TIME (Slide 8 Simulation Basis)
plt.figure(figsize=(12, 5))
plt.plot(times, attention_trajectory, color='#0284c7', linewidth=2, label=r'Attention Score ($\beta / \alpha$ Ratio)')
# Smoothed running average
running_avg = np.convolve(attention_trajectory, np.ones(10)/10, mode='same')
plt.plot(times, running_avg, color='#e11d48', linestyle='--', linewidth=2, label='Smoothed Trend Line')

plt.title('Dynamic Attention Ratios Over Learning Session', fontsize=14, fontweight='bold')
plt.xlabel('Time Elapsed (seconds)', fontsize=12)
plt.ylabel('Attention Intensity (Ratio Value)', fontsize=12)
plt.axhline(y=np.median(attention_trajectory), color='darkgreen', linestyle=':', label='Median Baseline Focus')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nProcess finished. All visual charts generated successfully!")